# NILM on PLAID Dataset

This notebook performs Non-Intrusive Load Monitoring (NILM) on the PLAID dataset to disaggregate the power consumption of Fridges and Microwaves from the aggregate signal.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.fft import fft, fftfreq
import glob
from tqdm import tqdm

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Paths
DATA_DIR = './Data'
META_2017 = os.path.join(DATA_DIR, 'meta_2017.json')
CSV_DIR = os.path.join(DATA_DIR, '2017')



## 1. Exploratory Data Analysis & Metadata Parsing
We first parse the metadata to find instances of Fridges and Microwaves.


In [ ]:
# Load metadata
with open(META_2017, 'r') as f:
    meta_data = json.load(f)

# Find Fridge and Microwave instances
fridge_ids = []
microwave_ids = []

for item in meta_data:
    app_type = item['meta']['appliance']['type']
    if app_type == 'Fridge':
        fridge_ids.append(item['id'])
    elif app_type == 'Microwave':
        microwave_ids.append(item['id'])

print(f"Total Fridge instances: {len(fridge_ids)}")
print(f"Total Microwave instances: {len(microwave_ids)}")

# Display a sample Fridge CSV
sample_csv = os.path.join(CSV_DIR, f"{fridge_ids[0]}.csv")
if os.path.exists(sample_csv):
    df_sample = pd.read_csv(sample_csv, header=None, names=['current', 'voltage'])
    print(f"Sample Fridge Data Shape: {df_sample.shape}")
    
    fig, ax1 = plt.subplots(figsize=(10, 4))
    color = 'tab:red'
    ax1.set_xlabel('Sample Index')
    ax1.set_ylabel('Voltage (V)', color=color)
    ax1.plot(df_sample['voltage'][:1000], color=color, alpha=0.7)
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Current (A)', color=color)
    ax2.plot(df_sample['current'][:1000], color=color, alpha=0.7)
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title('Fridge V & I Waveform Snippet (first 1000 samples)')
    fig.tight_layout()
    plt.show()
else:
    print(f"Could not find sample CSV: {sample_csv}")



## 2. Feature Engineering & Preprocessing
We will extract Vrms, Irms, Active Power (P), Reactive Power (Q), Apparent Power (S), Power Factor (PF), and Harmonics from sliding windows.


In [ ]:
def extract_features(v, i, fs=30000):
    """
    Extract NILM features from voltage and current windows.
    v: array of voltage samples in a window
    i: array of current samples in a window
    fs: sampling frequency (30 kHz for PLAID)
    """
    # Number of samples
    N = len(v)
    
    # Instantaneous power
    p_inst = v * i
    
    # Active Power (P)
    P = np.mean(p_inst)
    
    # Vrms and Irms
    Vrms = np.sqrt(np.mean(v**2))
    Irms = np.sqrt(np.mean(i**2))
    
    # Apparent Power (S)
    S = Vrms * Irms
    
    # Reactive Power (Q)
    # S^2 = P^2 + Q^2 => Q = sqrt(S^2 - P^2)
    Q = np.sqrt(max(S**2 - P**2, 0))
    
    # Power Factor (PF)
    PF = P / S if S > 0 else 0
    
    # FFT for Harmonics (Current)
    i_fft = np.abs(fft(i))
    freqs = fftfreq(N, 1/fs)
    
    # Focus on fundamental (60Hz for US dataset like PLAID) and odd harmonics
    # Since window might not be exactly aligned, we pick bins closest to 60, 180, 300, 420 Hz
    harmonics = []
    for h_freq in [60, 180, 300, 420, 540]:
        idx = np.argmin(np.abs(freqs[:N//2] - h_freq))
        harmonics.append(i_fft[idx] * 2.0 / N) # Normalize FFT amplitude
        
    features = [Vrms, Irms, P, Q, S, PF] + harmonics
    return np.array(features)

print("Feature extraction function defined.")



In [ ]:
def load_and_process_appliance_data(appliance_ids, label, window_size=30000):
    """
    Load CSVs, chunk into windows, and extract features.
    window_size=30000 means 1 second windows at 30kHz.
    """
    X_features = []
    Y_power = []
    
    # Limit to subset for demonstration to avoid MemoryError in notebook
    max_files = 50 
    
    for aid in tqdm(appliance_ids[:max_files], desc=f"Processing {label}"):
        filepath = os.path.join(CSV_DIR, f"{aid}.csv")
        if not os.path.exists(filepath):
            continue
            
        try:
            # Load CSV (no header)
            df = pd.read_csv(filepath, header=None, names=['current', 'voltage'])
            v = df['voltage'].values
            i = df['current'].values
            
            # Chunk into windows
            num_windows = len(v) // window_size
            for w in range(num_windows):
                start = w * window_size
                end = start + window_size
                
                v_win = v[start:end]
                i_win = i[start:end]
                
                feats = extract_features(v_win, i_win)
                
                # Target: we want to disaggregate power. We will use the Active Power (P) of the appliance 
                # as the target regression value. In this simplified setup, we treat the appliance's power 
                # as its label.
                p_win = np.mean(v_win * i_win)
                
                X_features.append(feats)
                Y_power.append(p_win)
        except Exception as e:
            print(f"Error reading {filepath}: {e}")
            
    return np.array(X_features), np.array(Y_power)

# In a real scenario, we'd aggregate multiple appliances to form an 'Aggregate' signal.
# For this demonstration, we'll build a model that takes aggregate features and predicts Fridge and Microwave power.
# We'll synthesize aggregate data by mixing Fridge and Microwave windows.

print("Loading Fridge data...")
X_fridge, y_fridge = load_and_process_appliance_data(fridge_ids, "Fridge")

print("Loading Microwave data...")
X_micro, y_micro = load_and_process_appliance_data(microwave_ids, "Microwave")



In [ ]:
# Synthesize Aggregate Signal
# We will create pairs of Fridge and Microwave windows to sum their features
min_samples = min(len(X_fridge), len(X_micro))

if min_samples == 0:
    print("Not enough data to synthesize. Check CSV directory.")
else:
    # We sum the raw time-domain signals to accurately reflect aggregate features
    X_agg_features = []
    Y_targets = []

    for idx in range(min_samples):
        # This is a simplification: adding the extracted features is NOT identical 
        # to adding raw waveforms and then extracting, but computing it from raw waveforms is better.
        # However, due to memory, we use this synthesized approach or we can just model them directly.
        # Let's assume the network takes the aggregate P, Q, Vrms, Irms and tries to predict Fridge P.
        
        # A simple linear sum for demonstration. In a true NILM pipeline, 
        # you sum raw V/I, then extract features from the sum.
        # We will just predict power from features.
        
        agg_feats = X_fridge[idx] + X_micro[idx]
        target = [y_fridge[idx], y_micro[idx]]
        
        X_agg_features.append(agg_feats)
        Y_targets.append(target)

    X = np.array(X_agg_features)
    Y = np.array(Y_targets)

    # Normalize X
    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0) + 1e-8
    X_norm = (X - X_mean) / X_std

    # Sequence generation (for CNN/LSTM)
    # We will use a sequence length of 5 windows
    seq_length = 5
    X_seq = []
    Y_seq = []

    for i in range(len(X_norm) - seq_length):
        X_seq.append(X_norm[i : i+seq_length])
        Y_seq.append(Y[i+seq_length - 1]) # Target is the power at the last step

    X_seq = np.array(X_seq)
    Y_seq = np.array(Y_seq)

    print(f"X_seq shape: {X_seq.shape}")
    print(f"Y_seq shape: {Y_seq.shape}")

    # Train/Val/Test Split
    n_samples = len(X_seq)
    train_end = int(0.7 * n_samples)
    val_end = int(0.85 * n_samples)

    X_train, y_train = X_seq[:train_end], Y_seq[:train_end]
    X_val, y_val = X_seq[train_end:val_end], Y_seq[train_end:val_end]
    X_test, y_test = X_seq[val_end:], Y_seq[val_end:]



## 3. Deep Learning Model (PyTorch 1D CNN)


In [ ]:
class NILM_CNN(nn.Module):
    def __init__(self, num_features, seq_length, num_targets):
        super(NILM_CNN, self).__init__()
        # Input shape: (batch, num_features, seq_length)
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.flatten = nn.Flatten()
        
        # Linear layers
        self.fc1 = nn.Linear(64 * seq_length, 128)
        self.fc2 = nn.Linear(128, num_targets)
        
    def forward(self, x):
        # PyTorch Conv1d expects (batch_size, channels, sequence_length)
        # Currently x is (batch_size, sequence_length, channels)
        x = x.transpose(1, 2)
        
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class NILMDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

if min_samples > 0:
    train_dataset = NILMDataset(X_train, y_train)
    val_dataset = NILMDataset(X_val, y_val)
    test_dataset = NILMDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    num_features = X_train.shape[2]
    model = NILM_CNN(num_features=num_features, seq_length=seq_length, num_targets=2)
    print(model)



## 4. Training Loop


In [ ]:
if min_samples > 0:
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    epochs = 15
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train MSE: {avg_train_loss:.2f} | Val MSE: {avg_val_loss:.2f}")

    # Plot loss
    plt.figure(figsize=(8,4))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.legend()
    plt.title('Training and Validation Loss')
    plt.show()



## 5. Evaluation


In [ ]:
if min_samples > 0:
    model.eval()
    predictions = []
    ground_truths = []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            predictions.append(outputs.numpy())
            ground_truths.append(y_batch.numpy())
            
    predictions = np.vstack(predictions)
    ground_truths = np.vstack(ground_truths)
    
    # Calculate MAE
    mae_fridge = np.mean(np.abs(predictions[:, 0] - ground_truths[:, 0]))
    mae_micro = np.mean(np.abs(predictions[:, 1] - ground_truths[:, 1]))
    
    print(f"Test MAE - Fridge: {mae_fridge:.2f} W")
    print(f"Test MAE - Microwave: {mae_micro:.2f} W")
    
    # Plotting first 100 predictions
    plot_len = min(100, len(predictions))
    
    fig, axs = plt.subplots(2, 1, figsize=(12, 8))
    
    axs[0].plot(ground_truths[:plot_len, 0], label='Ground Truth (Fridge)', color='blue')
    axs[0].plot(predictions[:plot_len, 0], label='Prediction (Fridge)', color='orange', linestyle='--')
    axs[0].set_title('Fridge Power Disaggregation')
    axs[0].set_ylabel('Power (W)')
    axs[0].legend()
    
    axs[1].plot(ground_truths[:plot_len, 1], label='Ground Truth (Microwave)', color='green')
    axs[1].plot(predictions[:plot_len, 1], label='Prediction (Microwave)', color='red', linestyle='--')
    axs[1].set_title('Microwave Power Disaggregation')
    axs[1].set_ylabel('Power (W)')
    axs[1].set_xlabel('Time Step')
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

